In [1]:
import csv
import re
from pathlib import Path
import pandas as pd


def parse_value(column, value):
    if value is None or value.strip() == "":
        return None

    value = value.strip()

    # Runtime may be stored as e.g. "0.123s"
    if column == "runTime":
        value = value.replace("s", "").strip()

    # Try integer
    try:
        return int(value)
    except ValueError:
        pass

    # Try float
    try:
        return float(value)
    except ValueError:
        pass

    # Otherwise keep it as a string
    return value


def readStatistics(folder):
    stats = {}

    # Read CSV files alphabetically
    csv_files = sorted(
        folder.glob("*.csv"),
        key=lambda p: p.name.lower()
    )

    for csv_file in csv_files:
        with open(csv_file) as f:
            reader = csv.DictReader(f, delimiter=";")

            for row in reader:
                instance_id = row["instanceId"]

                stats[instance_id] = {
                    column: parse_value(column, value)
                    for column, value in row.items()
                    if column != "instanceId"
                }

    return stats


def instance_sort_key(instance_id):
    match = re.match(r"([A-Za-z]+)(\d+)", instance_id)

    if match:
        prefix = match.group(1)
        number = int(match.group(2))

        return (prefix, number)

    return (instance_id, 0)


# --------------------------------------------------
# Find every folder containing CSV files
# --------------------------------------------------

methods = {}

folders = sorted(
    Path(".").iterdir(),
    key=lambda p: p.name.lower()
)

for folder in folders:
    if folder.is_dir() and any(folder.glob("*.csv")):
        methods[folder.name] = readStatistics(folder)


print("Found folders:")

for name in methods:
    print(" -", name)


# --------------------------------------------------
# Find every instance ID
# --------------------------------------------------

instance_ids = sorted(
    set().union(
        *(stats.keys() for stats in methods.values())
    ),
    key=instance_sort_key
)


# --------------------------------------------------
# Find every statistic present in any CSV file
# --------------------------------------------------

stat_names = set()

for stats in methods.values():
    for instance_stats in stats.values():
        stat_names.update(instance_stats.keys())


# --------------------------------------------------
# Create one table for every statistic
# --------------------------------------------------

tables = {}

for stat_name in sorted(stat_names):

    # Collect all non-missing values for this statistic
    available_values = [
        instance_stats[stat_name]
        for method_stats in methods.values()
        for instance_stats in method_stats.values()
        if stat_name in instance_stats
        and instance_stats[stat_name] is not None
    ]

    # Check whether this statistic contains only integers
    is_integer_stat = (
        len(available_values) > 0
        and all(isinstance(value, int) for value in available_values)
    )

    table_data = {}

    for method_name, method_stats in methods.items():
        values = []

        for instance_id in instance_ids:
            value = method_stats.get(
                instance_id,
                {}
            ).get(stat_name)

            # Missing integer statistic -> -1
            if value is None and is_integer_stat:
                value = -1

            values.append(value)

        table_data[method_name] = values


    table = pd.DataFrame(
        table_data,
        index=instance_ids
    )

    # Prevent pandas from turning integers into floats
    if is_integer_stat:
        table = table.astype(int)

    table.index.name = "Instance"

    tables[stat_name] = table

    print(f"\n{stat_name}")
    display(table)

Found folders:
 - 0_basic
 - 0_Hamza_1
 - 0_Hamza_2
 - 1_rotations
 - 2_area_order
 - 3_placement_lookahead
 - 4_order_lookahead
 - 5_different_orders
 - 6_placement_lookahead2
 - 9_winner_stats

nPlates


,0_basic,0_Hamza_1,0_Hamza_2,1_rotations,2_area_order,3_placement_lookahead,4_order_lookahead,5_different_orders,6_placement_lookahead2,9_winner_stats
Instance,,,,,,,,,,
A1,1,1,1,1,1,1,1,1,1,1
A2,7,11,10,6,6,5,5,5,5,5
A3,4,5,4,4,3,3,3,3,3,3
A4,5,5,4,4,3,3,3,3,3,3
A5,7,6,6,5,4,4,4,4,4,4
A6,4,5,5,4,4,3,3,3,3,3
A7,7,8,8,6,5,5,5,5,5,4
A8,13,15,15,11,10,9,9,9,9,8
A9,4,6,6,4,4,3,3,3,3,3



runTime


,0_basic,0_Hamza_1,0_Hamza_2,1_rotations,2_area_order,3_placement_lookahead,4_order_lookahead,5_different_orders,6_placement_lookahead2,9_winner_stats
Instance,,,,,,,,,,
A1,0.000054,None,None,0.000097,0.000128,0.018820,0.020851,0.061826,0.61926,None
A2,0.053026,None,None,0.084537,0.111745,3.221650,254.358000,23.545100,182.14700,None
A3,0.001798,None,None,0.000878,0.000764,0.485771,2.663250,3.105760,86.73280,None
A4,0.001129,None,None,0.004086,0.002750,0.779957,10.834500,7.662000,444.54800,None
A5,0.002978,None,None,0.002840,0.004538,1.358890,12.929200,9.672240,321.19100,None
A6,0.000961,None,None,0.000827,0.001180,0.161989,1.790110,0.878836,23.73030,None
A7,0.006669,None,None,0.002096,0.001735,0.390783,1.936580,2.033030,40.63130,None
A8,0.002462,None,None,0.009964,0.004422,1.066300,9.801980,11.145200,158.06800,None
A9,0.004067,None,None,0.005495,0.006512,0.529085,9.627550,5.224230,177.80700,None



totalGeoLoss


,0_basic,0_Hamza_1,0_Hamza_2,1_rotations,2_area_order,3_placement_lookahead,4_order_lookahead,5_different_orders,6_placement_lookahead2,9_winner_stats
Instance,,,,,,,,,,
A1,52.4205%,74%,74%,39.062%,39.062%,52.4205%,52.4205%,52.4205%,52.4205%,8%
A2,38.7968%,62.6364%,59.2%,32.6903%,28.197%,17.8746%,12.528%,17.7567%,12.7407%,6.6%
A3,43.7665%,55.4%,44.25%,32.5868%,24.8044%,18.2517%,25.7774%,21.9887%,18.2517%,5.33333%
A4,48.5646%,55.2%,44.25%,37.9663%,26.4191%,26.5809%,26.8901%,23.8813%,23.0354%,6.33333%
A5,53.8724%,50%,49.6667%,34.2644%,25.2944%,23.5608%,23.3947%,23.201%,15.6469%,5%
A6,37.9602%,54.2%,54.2%,34.8812%,32.7694%,24.7163%,25.1387%,22.8278%,21.3274%,7.66667%
A7,42.0125%,53.875%,53.75%,37.4334%,26.0353%,22.4331%,21.4947%,21.4975%,16.3601%,4.75%
A8,43.2078%,51.6667%,51.4667%,30.4018%,26.5818%,20.1056%,20.073%,19.5542%,18.4775%,5.5%
A9,40.8787%,60.5%,60.5%,39.308%,30.2101%,21.4285%,21.6443%,21.6838%,20.5399%,5.33333%



validSolution


,0_basic,0_Hamza_1,0_Hamza_2,1_rotations,2_area_order,3_placement_lookahead,4_order_lookahead,5_different_orders,6_placement_lookahead2,9_winner_stats
Instance,,,,,,,,,,
A1,1,1,1,1,1,1,1,1,1,1
A2,1,1,1,1,1,1,1,1,1,1
A3,1,1,1,1,1,1,1,1,1,1
A4,1,1,1,1,1,1,1,1,1,1
A5,1,1,1,1,1,1,1,1,1,1
A6,1,1,1,1,1,1,1,1,1,1
A7,1,1,1,1,1,1,1,1,1,1
A8,1,1,1,1,1,1,1,1,1,1
A9,1,1,1,1,1,1,1,1,1,1



widthResidual


,0_basic,0_Hamza_1,0_Hamza_2,1_rotations,2_area_order,3_placement_lookahead,4_order_lookahead,5_different_orders,6_placement_lookahead2,9_winner_stats
Instance,,,,,,,,,,
A1,3044,500,550,3692,3692,3044,3044,3044,3044,4461
A2,2704,500,505,269,2505,715,2505,757,2438,4584
A3,845,500,500,4685,684,2072,457,1309,2072,4153
A4,4685,500,500,3010,304,265,190,894,1082,4068
A5,3795,500,700,3191,410,945,995,1053,3108,5437
A6,2280,500,500,3307,3957,101,0,539,872,3520
A7,4289,500,500,1049,435,1808,2145,2144,3855,911
A8,2277,500,600,4210,1425,173,195,542,1248,2385
A9,352,500,500,964,3967,206,157,148,405,3189
